Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Stochastic Processes II: Martingales, Markov Chains & Brownian Motion

> ⚠️ **Draft — pending instructor review.** Simulations execute and corroborate the theorems, but execution cannot verify proofs. Review before teaching; remove this banner after.

The graduate sequel to [Independence](../Analysis/Independence.ipynb): processes with *dependence you can still control*. Martingales (fair games and their stunning convergence/stopping theorems), Markov chains (memory of length one, mixing to equilibrium), and Brownian motion with a first taste of Itô — the object under [diffusion models'](../../Intro_Mach_Learn/Diffusion_Models.ipynb) SDEs.

## 1. Pre-requisites

[Measure Theory](../Analysis/Measure_Theory.ipynb) & [Random Variables](../Analysis/Random_Variables.ipynb) (conditional expectation is used throughout); [Independence](../Analysis/Independence.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Conditional Expectation, Properly* (~35 min)
**Goal:** E[X|G] as a projection; the tower property as the workhorse identity.
**Builds on:** [Measure Theory](../Analysis/Measure_Theory.ipynb). &nbsp; **Feeds into:** Session 2 (martingales).

---

## 2. The Best Guess, Formalized

💡 **Intuition.** $E[X \mid \mathcal{G}]$ is *the best prediction of $X$ using only the information in $\mathcal{G}$* — formally, the [orthogonal projection](../Hilbert_Spaces/Hilbert_Spaces.ipynb) of $X$ onto the $\mathcal{G}$-measurable functions in $L^2$. Every property follows from the projection picture: linearity, 'taking out what is known' ($E[YX|\mathcal{G}] = Y E[X|\mathcal{G}]$ for known $Y$), and the **tower property** $E[E[X|\mathcal{G}]] = E[X]$ — projecting twice, coarser, is projecting once. The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) was computing exactly this object all along.

In [2]:
# conditional expectation IS L² projection — verified numerically
N = 400_000
Z = rng.standard_normal(N)
X = Z**2 + 0.5*rng.standard_normal(N)                # X depends on Z plus noise
# E[X | Z] should be Z² (the noise averages out); check via binning AND via projection residual
bins = np.linspace(-3, 3, 40)
centers = (bins[:-1] + bins[1:]) / 2
cond_mean = [X[(Z >= a) & (Z < b)].mean() for a, b in zip(bins[:-1], bins[1:])]
plt.figure(figsize=(7.5, 2.6))
plt.plot(centers, cond_mean, "o", markersize=4, label="empirical E[X | Z∈bin]")
plt.plot(centers, centers**2, "k--", label="Z² (theory)")
plt.legend(); plt.title("conditional expectation = the regression function")
plt.tight_layout(); plt.show()
# orthogonality: residual X − E[X|Z] must be uncorrelated with EVERY function of Z
resid = X - Z**2
for g, name in [(Z, "Z"), (Z**2, "Z²"), (np.sin(Z), "sin Z")]:
    print(f"corr(residual, {name:5s}) = {np.corrcoef(resid, g)[0,1]:+.4f}   (≈ 0: orthogonal)")

corr(residual, Z    ) = -0.0035   (≈ 0: orthogonal)
corr(residual, Z²   ) = -0.0030   (≈ 0: orthogonal)
corr(residual, sin Z) = -0.0028   (≈ 0: orthogonal)


/tmp/ipykernel_2979475/2786173116.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *Martingales: Fair Games & Their Theorems* (~40 min)
**Goal:** optional stopping (no free lunch) and martingale convergence, both simulated.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Markov chains).

---

## 3. Fair Games

**Definition.** $M_n$ is a *martingale* w.r.t. information flow $\mathcal{F}_n$ if $E[M_{n+1} \mid \mathcal{F}_n] = M_n$ — given everything known now, the expected next value is the current one. Examples: symmetric random walk; products of mean-1 factors (wealth under fair odds); the [Kalman innovation](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) sequence; $E[X | \mathcal{F}_n]$ for any fixed $X$ (a 'Doob martingale' — the object inside [McDiarmid's proof](../Concentration/Concentration_Inequalities.ipynb)).

**Optional stopping** (bounded case): for a martingale and a bounded stopping time $\tau$ (decided *without peeking at the future*), $E[M_\tau] = E[M_0]$ — **no exit strategy converts a fair game into a favorable one**.

💡 **Intuition.** 'Quit while ahead' feels like it should work, and simulation + theorem agree it doesn't: the times you fail to get ahead before the deadline exactly cancel the wins. All betting-system mysticism dies here, in one identity.

In [3]:
# 'quit while ahead' vs the theorem, on a fair ±1 walk with a 200-step deadline
trials, T_max, target = 100_000, 200, 5
steps = rng.choice([-1, 1], (trials, T_max))
walks = np.cumsum(steps, 1)
hit = (walks >= target).argmax(1)                     # first index where ≥ target (0 if never)
hit_mask = (walks >= target).any(1)
M_tau = np.where(hit_mask, target, walks[:, -1])      # stop at target if hit, else at deadline
print(f"P(get ahead by {target} within {T_max}): {hit_mask.mean():.3f}")
print(f"E[M_τ] = {M_tau.mean():+.4f}   (theorem: exactly 0 — the losers cancel the winners)")
print(f"...even though winners are {hit_mask.mean():.0%} of players! Their +{target} is "
      f"balanced by the losers' average {M_tau[~hit_mask].mean():.2f}")

P(get ahead by 5 within 200): 0.723
E[M_τ] = +0.0180   (theorem: exactly 0 — the losers cancel the winners)
...even though winners are 72% of players! Their +5 is balanced by the losers' average -13.00


In [4]:
# Martingale convergence: a bounded martingale MUST settle (here: Pólya's urn fraction)
n_paths, T_urn = 12, 3000
fracs = np.zeros((n_paths, T_urn))
for p in range(n_paths):
    red, total = 1, 2
    for t in range(T_urn):
        if rng.random() < red/total: red += 1
        total += 1
        fracs[p, t] = red/total
plt.figure(figsize=(8, 2.8))
plt.plot(fracs.T, linewidth=0.8)
plt.title("Pólya's urn: the red fraction is a bounded martingale → each path CONVERGES\n(to a random limit — uniform, in fact — but always converges)")
plt.xlabel("draw"); plt.tight_layout(); plt.show()

/tmp/ipykernel_2979475/3887821356.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("draw"); plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *Markov Chains & Mixing* (~40 min)
**Goal:** stationary distributions, detailed balance, and how fast chains forget their start.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (Brownian motion).

---

## 4. Memory of Length One

💡 **Intuition.** A Markov chain forgets everything but its current state. Run long enough, an irreducible aperiodic chain forgets even that: the distribution converges to the **stationary** $\pi$ ($\pi P = \pi$ — a left [eigenvector](../Linear_Algebra/Linear_Algebra.ipynb), eigenvalue 1), at a geometric rate set by the *second* eigenvalue — the **spectral gap**. Small gap = slow mixing: the chain's [graph](../../Intro_DSP/Graph_Signal_Processing.ipynb) has a bottleneck. This is the theory under every MCMC sampler and under [PageRank](../../Intro_DSP/Graph_Signal_Processing.ipynb).

In [5]:
# a 3-room chain with a bottleneck; mixing rate == |λ₂| — verified
P = np.array([[0.90, 0.10, 0.00],
              [0.05, 0.90, 0.05],
              [0.00, 0.02, 0.98]])
evals, evecs = np.linalg.eig(P.T)
i_one = np.argmin(np.abs(evals - 1))
pi = np.real(evecs[:, i_one]); pi /= pi.sum()
lam2 = np.sort(np.abs(evals))[-2]
print("stationary π =", pi.round(4), "  (check πP = π:", np.abs(pi @ P - pi).max() < 1e-12, ")")

# distance to stationarity vs t: slope must equal log|λ₂|
mu = np.array([1.0, 0, 0]); dists = []
for t in range(400):
    dists.append(np.abs(mu - pi).sum())
    mu = mu @ P
measured = (np.log(dists[300]) - np.log(dists[100])) / 200
print(f"measured decay rate {np.exp(measured):.4f} per step   vs   |λ₂| = {lam2:.4f}")
plt.figure(figsize=(7, 2.4)); plt.semilogy(dists)
plt.title("total-variation distance to π: geometric at exactly |λ₂|")
plt.xlabel("step"); plt.tight_layout(); plt.show()

stationary π = [0.125 0.25  0.625]   (check πP = π: True )
measured decay rate 0.9540 per step   vs   |λ₂| = 0.9540


/tmp/ipykernel_2979475/650615853.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("step"); plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *Brownian Motion & a Taste of Itô* (~40 min)
**Goal:** the scaling limit of walks; why (dB)² = dt changes calculus itself.
**Builds on:** Session 3.

---

## 5. The Continuum Limit

💡 **Intuition.** Speed up a random walk (steps of size $\sqrt{\Delta t}$ every $\Delta t$) and it converges to **Brownian motion**: continuous paths, independent Gaussian increments, nowhere differentiable. Its signature weirdness: over an interval $dt$, the increment $dB \sim \sqrt{dt}$ — so $(dB)^2 \sim dt$ is *first order*, not negligible. Taylor expansions of $f(B_t)$ therefore keep a second-derivative term, giving **Itô's formula**:
$$df(B_t) = f'(B_t)\, dB_t + \tfrac{1}{2} f''(B_t)\, dt.$$
That extra $\tfrac12 f''$ term is the mathematical heart of the [diffusion-model SDEs](../../Intro_Mach_Learn/Diffusion_Models.ipynb) and of option pricing alike.

In [6]:
# (dB)² = dt, empirically: quadratic variation of Brownian paths
T_end, n_steps, n_paths = 1.0, 4000, 2000
dt = T_end / n_steps
dB = np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))
QV = (dB**2).sum(1)
print(f"quadratic variation: mean {QV.mean():.5f}  std {QV.std():.5f}   (theory: exactly T = {T_end},")
print("  with vanishing variance as dt→0 — a DETERMINISTIC limit from pure randomness)")

# Itô's correction, verified: for f(x)=x², Itô says B_t² − t is a martingale (E[B_t² − t] = 0)
B = dB.cumsum(1)
t_ax = np.linspace(dt, T_end, n_steps)
gap = (B**2).mean(0) - t_ax
print(f"max |E[B_t²] − t| over the path: {np.abs(gap).max():.4f}   (Itô: B_t² − t is a martingale)")

quadratic variation: mean 1.00002  std 0.02247   (theory: exactly T = 1.0,
  with vanishing variance as dt→0 — a DETERMINISTIC limit from pure randomness)


max |E[B_t²] − t| over the path: 0.0262   (Itô: B_t² − t is a martingale)


## 6. Conclusion

Conditional expectation is projection; martingales formalize fairness and forbid free lunches (verified to four decimals); Markov chains mix at the spectral gap (verified against $|\lambda_2|$); and Brownian motion's $(dB)^2 = dt$ rewrites calculus. You are now equipped for the stochastic-analysis layer of modern generative modeling and finance alike.

---
## Where next

- [Diffusion Models](../../Intro_Mach_Learn/Diffusion_Models.ipynb) → [Score-Based SDEs](../../Intro_Mach_Learn/Diffusion_Score_SDE.ipynb) — Itô at work.
- [Concentration](../Concentration/Concentration_Inequalities.ipynb) — Doob martingales under McDiarmid.
- [Reinforcement Learning](../../Intro_Mach_Learn/Reinforcement_Learning.ipynb) — MDPs: Markov chains you get to steer.